# No Window Preprocessing メモリ使用量デバッグ

メモリ不足の原因を特定するための段階的実行ノートブック

In [1]:
import sys
import os
from pathlib import Path
import psutil
import gc
import numpy as np
import pandas as pd
import yaml
import logging

# プロジェクトルートをパスに追加
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils.pipeline import Preprocessor
from src.utils.logging_utils import setup_logging
from src.utils.preprocessing import augment_handedness_flip

# ログ設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
def get_memory_usage():
    """現在のメモリ使用量を取得"""
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()
    return {
        'rss_mb': memory_info.rss / 1024 / 1024,  # MB
        'vms_mb': memory_info.vms / 1024 / 1024,  # MB
        'percent': process.memory_percent()
    }

def log_memory_usage(stage: str):
    """メモリ使用量をログ出力"""
    mem = get_memory_usage()
    logger.info(f"{stage}: RSS={mem['rss_mb']:.1f}MB, VMS={mem['vms_mb']:.1f}MB, {mem['percent']:.1f}%")
    return mem

In [3]:
# 設定ファイル読み込み
config_path = Path("../config/config_v50.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

logger.info(f"Config loaded: {config_path}")
log_memory_usage("Config loaded")

INFO:__main__:Config loaded: ../config/config_v50.yaml
INFO:__main__:Config loaded: RSS=228.9MB, VMS=1944.6MB, 1.2%


{'rss_mb': 228.90234375, 'vms_mb': 1944.6015625, 'percent': 1.1515856484825047}

In [4]:
# データ読み込み
data_dir = Path("../data")
logger.info("Loading train.csv...")
train_df = pd.read_csv(data_dir / "train.csv")
logger.info(f"Train data shape: {train_df.shape}")
log_memory_usage("Train data loaded")

# メモリ使用量の詳細
logger.info(f"Train data memory usage: {train_df.memory_usage(deep=True).sum() / 1024 / 1024:.1f}MB")

INFO:__main__:Loading train.csv...
INFO:__main__:Train data shape: (574945, 341)
INFO:__main__:Train data loaded: RSS=3246.9MB, VMS=4962.4MB, 16.3%
INFO:__main__:Train data memory usage: 1774.6MB


In [5]:
# Demographicsデータ読み込み
logger.info("Loading demographics data...")
train_demo = pd.read_csv(data_dir / "train_demographics.csv")
logger.info(f"Demographics shape: {train_demo.shape}")
log_memory_usage("Demographics loaded")

# データマージ
logger.info("Merging data...")
train_df = train_df.merge(train_demo, on="subject", how="left")
logger.info(f"Merged data shape: {train_df.shape}")
log_memory_usage("Data merged")

INFO:__main__:Loading demographics data...
INFO:__main__:Demographics shape: (81, 8)
INFO:__main__:Demographics loaded: RSS=3246.9MB, VMS=4962.4MB, 16.3%
INFO:__main__:Merging data...
INFO:__main__:Merged data shape: (574945, 348)
INFO:__main__:Data merged: RSS=3248.0MB, VMS=4962.4MB, 16.3%


{'rss_mb': 3248.0390625, 'vms_mb': 4962.41015625, 'percent': 16.34057174255372}

In [ ]:
# NoWindowPreprocessor初期化
logger.info("Initializing NoWindowPreprocessor...")
from scripts.run_preprocessing_no_windows import NoWindowPreprocessor

pp = NoWindowPreprocessor(config)
log_memory_usage("Preprocessor initialized")

INFO:__main__:Initializing NoWindowPreprocessor...
INFO:__main__:Preprocessor initialized: RSS=3248.0MB, VMS=4962.4MB, 16.3%


{'rss_mb': 3248.0390625, 'vms_mb': 4962.41015625, 'percent': 16.34057174255372}

: 

In [ ]:
# データクリーニング段階
logger.info("Starting data cleaning...")
df_proc = pp._maybe_clean(train_df)
logger.info(f"Cleaned data shape: {df_proc.shape}")
log_memory_usage("Data cleaning completed")

# メモリ解放
del train_df
gc.collect()
log_memory_usage("Original data freed")

INFO:__main__:Starting data cleaning...
/mnt/c/Users/ShunK/works/CMI_comp/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().all(axis=1)
/mnt/c/Users/ShunK/works/CMI_comp/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().all(axis=1)
/mnt/c/Users/ShunK/works/CMI_comp/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` ma

interp:   0%|          | 0/8151 [00:00<?, ?seq/s]

concat:   0%|          | 0/8151 [00:00<?, ?it/s]

In [ ]:
# Handedness augmentation
if pp.use_handedness_augmentation:
    logger.info("Applying handedness augmentation...")
    df_proc = augment_handedness_flip(df_proc)
    logger.info(f"Augmented data shape: {df_proc.shape}")
    log_memory_usage("Handedness augmentation completed")
else:
    logger.info("Skipping handedness augmentation")

In [ ]:
# シーケンス抽出（段階的）
logger.info("Starting sequence extraction...")

# センサー列の取得
sensor_cols = (
    config.get("sensor_acc_cols", [])
    + config.get("sensor_rot_cols", [])
    + config.get("sensor_thm_cols", [])
)
if pp.use_world_acc:
    world_acc_cols = [f"acc_w_{ax}" for ax in "xyz"] + [f"lin_acc_{ax}" for ax in "xyz"]
    sensor_cols.extend(world_acc_cols)

demographics_cols = config.get("demographics_cols", [])

logger.info(f"Sensor columns: {len(sensor_cols)}")
logger.info(f"Demographics columns: {len(demographics_cols)}")
log_memory_usage("Columns identified")

In [ ]:
# グループ化してシーケンス抽出（メモリ効率化）
logger.info("Extracting sequences...")
sequences = []
demographics = []
labels = []

# メモリ効率化のため、グループ化して処理
grouped = df_proc.groupby(['subject', 'sequence_id'])
total_groups = len(grouped)
logger.info(f"Processing {total_groups} sequences...")

# 最初の1000シーケンスのみでテスト
test_limit = min(1000, total_groups)
logger.info(f"Testing with first {test_limit} sequences...")

for i, ((subject, sequence_id), seq_data) in enumerate(grouped):
    if i >= test_limit:
        break
        
    if i % 100 == 0:
        logger.info(f"Processing sequence {i}/{test_limit}")
        log_memory_usage(f"Sequence {i}")
    
    # センサーデータを抽出（時系列）
    sensor_data = seq_data[sensor_cols].values
    
    # 人口統計データ（シーケンス単位で平均）
    demo_data = seq_data[demographics_cols].mean().values
    
    # ラベル
    label = seq_data['gesture'].iloc[0] if 'gesture' in seq_data.columns else -1
    
    sequences.append(sensor_data)
    demographics.append(demo_data)
    labels.append(label)

logger.info(f"Extracted {len(sequences)} sequences")
log_memory_usage("Sequence extraction completed")

In [ ]:
# センサーデータ処理（段階的）
logger.info("Processing sensor data...")
X_sensor_clean = []
all_sensor_data_chunks = []

for i, seq in enumerate(sequences):
    if i % 100 == 0:
        logger.info(f"Processing sensor data {i}/{len(sequences)}")
        log_memory_usage(f"Sensor processing {i}")
    
    # 欠損値処理
    seq_clean = np.nan_to_num(seq, nan=0.0)
    # 外れ値処理（シーケンス単位）
    seq_clean = pp._handle_outliers_in_sensor_data(seq_clean)
    X_sensor_clean.append(seq_clean)
    
    # メモリ効率化のため、チャンク単位で正規化パラメータを計算
    if len(all_sensor_data_chunks) < 100:  # 100シーケンスごとにチャンク
        all_sensor_data_chunks.append(seq_clean)
    else:
        # チャンクを結合して正規化パラメータを更新
        chunk_data = np.vstack(all_sensor_data_chunks)
        if not hasattr(pp, '_partial_sensor_scaler'):
            from sklearn.preprocessing import StandardScaler
            pp._partial_sensor_scaler = StandardScaler()
            pp._partial_sensor_scaler.partial_fit(chunk_data)
        else:
            pp._partial_sensor_scaler.partial_fit(chunk_data)
        all_sensor_data_chunks = [seq_clean]  # リセット

logger.info("Sensor data processing completed")
log_memory_usage("Sensor processing completed")

In [ ]:
# 最後のチャンク処理
if all_sensor_data_chunks:
    chunk_data = np.vstack(all_sensor_data_chunks)
    if hasattr(pp, '_partial_sensor_scaler'):
        pp._partial_sensor_scaler.partial_fit(chunk_data)
        pp.sensor_scaler = pp._partial_sensor_scaler
    else:
        pp.sensor_scaler.fit(chunk_data)
    logger.info("Final chunk processed")
else:
    # 部分的な正規化ができない場合は全データで学習
    all_sensor_data = np.vstack(X_sensor_clean)
    pp.sensor_scaler.fit(all_sensor_data)
    logger.info("Full data normalization completed")

log_memory_usage("Normalization completed")

In [ ]:
# 人口統計データ処理
logger.info("Processing demographics data...")
X_demo_clean = np.nan_to_num(np.array(demographics), nan=0.0)
pp.demo_scaler.fit(X_demo_clean)
logger.info(f"Demographics shape: {X_demo_clean.shape}")
log_memory_usage("Demographics processing completed")

In [ ]:
# 表形式特徴量抽出（段階的）
logger.info("Extracting tabular features...")
tabular_features = []

for i, seq in enumerate(sequences):
    if i % 100 == 0:
        logger.info(f"Extracting tabular features {i}/{len(sequences)}")
        log_memory_usage(f"Tabular extraction {i}")
    
    # 基本統計量
    seq_features = []
    for col_idx, col_name in enumerate(sensor_cols):
        values = seq[:, col_idx]
        seq_features.extend([
            np.mean(values),  # mean
            np.std(values),   # std
            np.max(values) - np.min(values),  # range
            np.sqrt(np.mean(values**2)),  # rms
            np.sum(values**2),  # energy
        ])
    
    # ピーク特徴量
    for col_idx in range(len(sensor_cols)):
        values = seq[:, col_idx]
        # ピーク検出（簡易版）
        peaks = 0
        for j in range(1, len(values) - 1):
            if values[j] > values[j-1] and values[j] > values[j+1]:
                peaks += 1
        seq_features.append(peaks)
    
    # FFT特徴量（オプション）
    pp_config = config.get("preprocessing", {})
    fft_bands = pp_config.get("fft_bands", [])
    if fft_bands:
        for col_idx in range(len(sensor_cols)):
            values = seq[:, col_idx]
            # FFT計算
            fft_vals = np.fft.fft(values)
            fft_power = np.abs(fft_vals)**2
            
            for low, high in fft_bands:
                # 周波数帯域のエネルギー
                freq_bins = np.fft.fftfreq(len(values))
                mask = (freq_bins >= low) & (freq_bins <= high)
                energy = np.sum(fft_power[mask])
                seq_features.append(energy)
    
    tabular_features.append(seq_features)

logger.info(f"Tabular features shape: {len(tabular_features)} x {len(tabular_features[0])}")
log_memory_usage("Tabular features extraction completed")

In [ ]:
# 表形式特徴量の正規化
logger.info("Normalizing tabular features...")
tabular_features = np.array(tabular_features)
tab_clean = np.nan_to_num(tabular_features, nan=0.0)

# クリップ処理の閾値を設定
pp.tab_clip_low = np.percentile(tab_clean, 0.5)
pp.tab_clip_high = np.percentile(tab_clean, 99.5)

pp.tab_scaler.fit(tab_clean)
logger.info(f"Tabular features normalized: {tab_clean.shape}")
log_memory_usage("Tabular normalization completed")

In [ ]:
# ラベルエンコーディング
logger.info("Processing labels...")
labels = np.array(labels)
if len(np.unique(labels)) > 1 and not (len(np.unique(labels)) == 1 and labels[0] == -1):
    from sklearn.preprocessing import LabelEncoder
    pp.label_encoder = LabelEncoder()
    pp.label_encoder.fit(labels)
    logger.info(f"Label encoder created: {list(pp.label_encoder.classes_)}")
else:
    logger.info("No label encoding needed")

pp._fitted = True
logger.info("NoWindowPreprocessor fitting completed")
log_memory_usage("Fitting completed")

In [ ]:
# メモリ使用量の最終確認
logger.info("=== Final Memory Usage ===")
final_mem = log_memory_usage("Final")

# システム全体のメモリ状況
system_mem = psutil.virtual_memory()
logger.info(f"System memory: {system_mem.available / 1024 / 1024 / 1024:.1f}GB available out of {system_mem.total / 1024 / 1024 / 1024:.1f}GB")
logger.info(f"System memory usage: {system_mem.percent:.1f}%")

# 結果サマリー
logger.info("=== Processing Summary ===")
logger.info(f"Sequences processed: {len(sequences)}")
logger.info(f"Sensor features: {len(sensor_cols)}")
logger.info(f"Demographics features: {len(demographics_cols)}")
logger.info(f"Tabular features: {len(tabular_features[0])}")
logger.info("Processing completed successfully!")